# 01 — A Simple Vanilla RNN from Scratch (PyTorch)

This notebook builds a minimal character-level RNN that learns to predict the next character in a short piece of text. The goal is to see the core recurrence mechanism in action, not to build a production model.

**What you'll see:**
- How to build a vanilla RNN using `nn.RNN`
- How hidden state carries information across timesteps
- How to train with backpropagation through time (handled automatically by autograd)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

## 1. Prepare a tiny character-level dataset

In [ ]:
text = "hello world, recurrent neural networks are fun to learn"

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {chars}")

In [ ]:
# Build (input, target) pairs: predict the next character given the current one
seq_len = 5

def encode(s):
    return [char_to_idx[c] for c in s]

data = encode(text)

inputs, targets = [], []
for i in range(len(data) - seq_len):
    inputs.append(data[i:i + seq_len])
    targets.append(data[i + 1:i + seq_len + 1])

inputs = torch.tensor(inputs)
targets = torch.tensor(targets)
print(inputs.shape, targets.shape)

## 2. Define the RNN model

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)              # (batch, seq_len, embed_dim)
        output, hidden = self.rnn(embedded, hidden)  # output: (batch, seq_len, hidden_dim)
        logits = self.fc(output)                   # (batch, seq_len, vocab_size)
        return logits, hidden

model = CharRNN(vocab_size)
print(model)

## 3. Train the model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 200
for epoch in range(epochs):
    optimizer.zero_grad()
    logits, _ = model(inputs)
    loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # prevent exploding gradients
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")

## 4. Generate text with the trained model

In [ ]:
def generate(model, start_str, length=40):
    model.eval()
    chars_generated = list(start_str)
    input_seq = torch.tensor([encode(start_str)])
    hidden = None

    with torch.no_grad():
        for _ in range(length):
            logits, hidden = model(input_seq, hidden)
            next_idx = torch.argmax(logits[0, -1]).item()
            chars_generated.append(idx_to_char[next_idx])
            input_seq = torch.tensor([[next_idx]])

    return "".join(chars_generated)

print(generate(model, "hello", length=40))

## Try it yourself

- Swap `nn.RNN` for `nn.LSTM` or `nn.GRU` and compare how quickly the loss drops.
- Increase `hidden_dim` and see how it affects generation quality.
- Try a longer, more varied piece of text as training data.
- Print the hidden state shape at each step to see how information is carried forward.